# Visualize ORA

In [9]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from collections import defaultdict, Counter
from upsetplot import UpSet, from_contents
from PIL import Image

from config import PIPELINE_RUN_DIR, AMIMS, NETWORKS, SEED_SETS
from functions import split_module_id, save_figure 
prefix = "09"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
base_dir = os.path.join(PIPELINE_RUN_DIR, "main/results/evaluation/gprofiler")
amim_meta_df = pd.DataFrame.from_dict(AMIMS)
network_meta_df = pd.DataFrame.from_dict(NETWORKS)
seed_meta_df = pd.DataFrame.from_dict(SEED_SETS)

In [4]:
# Create a df containing the paths to all gprofiler results + meta data (seed, network, amim)
path_dict = {}
for root, dirs, files in os.walk(base_dir):
    for filename in files:
        # remove the suffix from the filename to get the id
        id = filename.replace(".gprofiler2.all_enriched_pathways.tsv","")
        path_dict[id] = os.path.join(root, filename)
df = pd.DataFrame.from_dict(path_dict, orient="index", columns=["path"])
df[["seed_id","network_id","amim_id"]] = df.index.to_series().apply(split_module_id)
df["network"] = df["network_id"].replace(dict(zip(network_meta_df.id, network_meta_df.label)))
df["amim"] = df["amim_id"].replace(dict(zip(amim_meta_df.id, amim_meta_df.label)))
df

,path,seed_id,network_id,amim_id,network,amim
HD.string.human_links_v12_0_min700.Symbol.domino,../pipeline_runs/main/results/evaluation/gprof...,HD,string.human_links_v12_0_min700.Symbol,domino,STRING (high confidence),DOMINO
LUAD.string.human_links_v12_0_min700.Symbol.robust,../pipeline_runs/main/results/evaluation/gprof...,LUAD,string.human_links_v12_0_min700.Symbol,robust,STRING (high confidence),ROBUST
UC.string.human_physical_links_v12_0_min700.Symbol.no_tool,../pipeline_runs/main/results/evaluation/gprof...,UC,string.human_physical_links_v12_0_min700.Symbol,no_tool,"STRING (physical, high confidence)",Only seeds
LUAD.string.human_links_v12_0_min900.Symbol.firstneighbor,../pipeline_runs/main/results/evaluation/gprof...,LUAD,string.human_links_v12_0_min900.Symbol,firstneighbor,STRING (highest confidence),1st Neighbors
CD.nedrex.reviewed_proteins_exp_high_confidence.Symbol.domino,../pipeline_runs/main/results/evaluation/gprof...,CD,nedrex.reviewed_proteins_exp_high_confidence.S...,domino,NeDRex (high confidence),DOMINO
...,...,...,...,...,...,...
HD.biogrid.4_4_242_homo_sapiens.Symbol.robust_bias_aware,../pipeline_runs/main/results/evaluation/gprof...,HD,biogrid.4_4_242_homo_sapiens.Symbol,robust_bias_aware,BioGRID,ROBUST\n(bias-aware)
ALS.iid.human.Symbol.domino,../pipeline_runs/main/results/evaluation/gprof...,ALS,iid.human.Symbol,domino,IID,DOMINO
LUAD.nedrex.reviewed_proteins_exp.Symbol.robust_bias_aware,../pipeline_runs/main/results/evaluation/gprof...,LUAD,nedrex.reviewed_proteins_exp.Symbol,robust_bias_aware,NeDRex,ROBUST\n(bias-aware)
CD.nedrex.reviewed_proteins_exp.Symbol.firstneighbor,../pipeline_runs/main/results/evaluation/gprof...,CD,nedrex.reviewed_proteins_exp.Symbol,firstneighbor,NeDRex,1st Neighbors


In [5]:
def extract_data(ids, list_enriched_pathways):
    """
    Extract pathway data from a list of enriched pathways files.
    
    Args:
        ids (list): A list of module IDs.
        list_enriched_pathways (list): A list of paths to TSV files containing enriched pathways.

    Returns:
        dict: A dictionary mapping each source of pathways to the dictionary {module ID: set(enriched pathways)}.
    """
    
    unique_sources = []
    
    # create a dictionary {source: {id:set(pathways)}}
    d_source_pathways = defaultdict(lambda: defaultdict(set))
    
    for (id, pathways) in zip(ids, list_enriched_pathways):
        
        df_pathways = pd.read_csv(pathways, sep="\t")

        assert set(["source", "term_name"]).issubset(df_pathways.columns)
        
        # Get the sources
        if len(unique_sources) == 0: 
            unique_sources = df_pathways['source'].unique()

        # Get the pathways for each source and each module 
        for source in unique_sources:
            d_source_pathways[source][id] = set(df_pathways[df_pathways['source'] == source]['term_name'])

    return unique_sources, d_source_pathways

def frequency_pathways_in_modules(ids, list_enriched_pathways):
    """
    Given a list of modules, return a dictionary mapping each pathway to the frequency of modules in which it appears.

    Input:
        modules: List of original modules (each module is a set of genes).
        module_names: List of names for the original modules.
    Returns:
        dict: {gene: [module1, module2, ...]} where module1, module2 are modules containing the gene.
    """
    
    d_pathways = defaultdict(list)
    for id, pathways in zip(ids, list_enriched_pathways):
        for p in pathways:
            d_pathways[p].append(id)

    # sort dictionary with descending frequencies
    sorted_pathway_frequency = dict(sorted(d_pathways.items(), key=lambda item: len(item[1]), reverse=True))

    return sorted_pathway_frequency


# Upset plots

In [ ]:
# group by network and seed id
for (network, seed_id), group in df[df["seed_id"]=="CD"].groupby(["network", "seed_id"]):
    print(f"Processing network: {network}, seed_id: {seed_id}")
    unique_sources, d_source_pathways = extract_data(group.amim, group.path)

    tmp_files = []

    for i, source in enumerate(unique_sources):

        id = f"{network}_{seed_id}_{source}"
        
        # The values of module_map are the sets of pathways for this source
        module_map = d_source_pathways[source]
        upset_data = from_contents(module_map)
        
        fig = plt.figure(figsize=(8, 6))  # workaround: upset needs a Figure
        upset_plot = UpSet(upset_data, subset_size='count', show_counts=True, sort_by='cardinality', max_subset_rank=20)
        upset_plot.plot(fig = fig)
        fig.suptitle(f"Distribution of {source} terms in Disease Modules")

        # save the figure for later use
        tmp_file = f"_tmp_upset_{id}.png"
        fig.savefig(tmp_file, dpi=300, bbox_inches='tight')
        tmp_files.append(tmp_file)
        plt.close(fig)  

    # Save the entire figure with all subplots
    images = [Image.open(f) for f in tmp_files]
    widths, heights = zip(*(img.size for img in images))

    total_height = sum(heights)
    max_width = max(widths)

    combined = Image.new("RGB", (max_width, total_height), "white")

    y_offset = 0
    for img in images:
        combined.paste(img, (0, y_offset))
        y_offset += img.size[1]

    combined.save(f"{prefix}_pathway_overlap_{seed_id}_{network}.png")
    combined.save(f"{prefix}_pathway_overlap_{seed_id}_{network}.pdf")

    # remove temporary files
    for f in tmp_files:
        os.remove(f)


Processing network: BioGRID, seed_id: CD


/home/johannes/micromamba/envs/mdp_demonstration/lib/python3.12/site-packages/upsetplot/data.py:385: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(False, inplace=True)
/home/johannes/micromamba/envs/mdp_demonstration/lib/python3.12/site-packages/upsetplot/plotting.py:795: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the orig

Processing network: HIPPIE (high confidence), seed_id: CD


/home/johannes/micromamba/envs/mdp_demonstration/lib/python3.12/site-packages/upsetplot/data.py:385: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(False, inplace=True)
/home/johannes/micromamba/envs/mdp_demonstration/lib/python3.12/site-packages/upsetplot/plotting.py:795: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the orig

Processing network: HIPPIE (medium confidence), seed_id: CD


...

/home/johannes/micromamba/envs/mdp_demonstration/lib/python3.12/site-packages/upsetplot/data.py:385: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(False, inplace=True)
/home/johannes/micromamba/envs/mdp_demonstration/lib/python3.12/site-packages/upsetplot/plotting.py:795: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the orig

Processing network: STRING (physical, high confidence), seed_id: CD


/home/johannes/micromamba/envs/mdp_demonstration/lib/python3.12/site-packages/upsetplot/data.py:385: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(False, inplace=True)
/home/johannes/micromamba/envs/mdp_demonstration/lib/python3.12/site-packages/upsetplot/plotting.py:795: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the orig

Processing network: STRING (physical, highest confidence), seed_id: CD


/home/johannes/micromamba/envs/mdp_demonstration/lib/python3.12/site-packages/upsetplot/data.py:385: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(False, inplace=True)
/home/johannes/micromamba/envs/mdp_demonstration/lib/python3.12/site-packages/upsetplot/plotting.py:795: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the orig

In [ ]:

# Compute frequencies of pathways in modules
pathway_frequency = frequency_pathways_in_modules(group.index, module_map.values())

# write multiqc summary
with open(f"{id}_terms_frequency_in_modules.tsv", "w") as f:
    f.write("term/pathway\tfrequency in modules\tmodules\n")
    for term, modules in pathway_frequency.items():
        f.write(f"{term}\t{len(modules)}/{len(group.index)}\t{modules}\n")